In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import os
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.model_selection import GridSearchCV, StratifiedShuffleSplit
from xgboost.sklearn import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from scipy.special import entr
from sklearn import preprocessing
import matplotlib.pyplot as plt
import math
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import accuracy_score,confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from IPython.display import display
#import itertools
#from itables import init_notebook_mode
import random
#init_notebook_mode(all_interactive=True)


In [9]:
def modify_timestamps(dfs):
    modified_dfs = []
    for df in dfs:
        if not df.empty:
            df['timestamp_modified'] = df['timestamp'].iloc[0] + df.index * 10
        modified_dfs.append(df)
    return modified_dfs

In [37]:
# Using pickle, load modification_timestamp.pkl file
# Path: timestamp_modify_to_make_it_one.ipynb
import pickle
with open('modification_timestamp.pkl', 'rb') as f:
    ttt = pickle.load(f)



In [42]:
del ttt

In [3]:
import pickle
# Load the dictionary back from the pickle file.
with open('user_dict_modified_timestamp.pkl', 'rb') as f:
    user_dict = pickle.load(f)

In [43]:
del user_dict['18']

In [46]:
# Create temporary dictionary to store modified dataframes
temp_dict = {}
# For loop that calls mofify_timestamp function on each list of user's dataframe
for user in user_dict:
    temp_dict[user] = modify_timestamps(user_dict[user])

# Check first five rows for each user
for user in temp_dict:
    print(user)
    print(temp_dict[user][0].head())
    print()





1
     client_timestamp    x    y button state    window\n      timestamp  \
0      1570169899.971  694  489   None  Move  browsing\n  1570169899971   
1       1570169899.99  703  473   None  Move  browsing\n  1570169899990   
2  1570169900.0089998  713  454   None  Move  browsing\n  1570169900008   
3      1570169900.028  725  436   None  Move  browsing\n  1570169900028   
4      1570169900.046  737  417   None  Move  browsing\n  1570169900046   

   timestamp_modified  
0       1570169899971  
1       1570169899981  
2       1570169899991  
3       1570169900001  
4       1570169900011  

10
     client_timestamp     x    y button state    window\n      timestamp  \
0  1570172649.4120002   960  542   None  Move  browsing\n  1570172649412   
1       1570172649.42   984  636   None  Move  browsing\n  1570172649420   
2      1570172649.436   994  719   None  Move  browsing\n  1570172649436   
3      1570172649.444   997  748   None  Move  browsing\n  1570172649444   
4      1570172649.4

In [47]:
temp_dict.keys()

dict_keys(['1', '10', '11', '12', '13', '14', '15', '16', '17', '19', '2', '3', '4', '5', '6', '7', '8', '9'])

In [48]:
# Write temp_dict to pickle file called modification_timestamp.pkl
with open('modification_timestamp.pkl', 'wb') as f:
    pickle.dump(temp_dict, f)




In [52]:
# delete user 6
del temp_dict['6']

In [54]:
timestamp_diff_dict = {}

for key, df_list in temp_dict.items():
    df_timestamp_diffs = []
    for df in df_list:
        # print key and column names with clear indication of which DataFrame we are looking at
        # this is useful for debugging


        print(key)
        print(df.columns)
        print("-----")
        first_timestamp = df['timestamp_modified'].iloc[0]
        last_timestamp = df['timestamp_modified'].iloc[-1]
        timestamp_difference = last_timestamp - first_timestamp
        # Convert to seconds
        timestamp_difference = timestamp_difference / 1000

        df_timestamp_diffs.append(timestamp_difference)
    
    timestamp_diff_dict[key] = df_timestamp_diffs

# Now, timestamp_diff_dict contains the timestamp difference for each DataFrame in the list for each key
print(timestamp_diff_dict)

1
Index(['client_timestamp', 'x', 'y', 'button', 'state', 'window\n',
       'timestamp', 'timestamp_modified'],
      dtype='object')
-----
1
Index(['client_timestamp', 'x', 'y', 'button', 'state', 'window\n',
       'timestamp', 'timestamp_modified'],
      dtype='object')
-----
1
Index(['client_timestamp', 'x', 'y', 'button', 'state', 'window\n',
       'timestamp', 'timestamp_modified'],
      dtype='object')
-----
1
Index(['client_timestamp', 'x', 'y', 'button', 'state', 'window\n',
       'timestamp', 'timestamp_modified'],
      dtype='object')
-----
1
Index(['client_timestamp', 'x', 'y', 'button', 'state', 'window\n',
       'timestamp', 'timestamp_modified'],
      dtype='object')
-----
1
Index(['client_timestamp', 'x', 'y', 'button', 'state', 'window\n',
       'timestamp', 'timestamp_modified'],
      dtype='object')
-----
1
Index(['client_timestamp', 'x', 'y', 'button', 'state', 'window\n',
       'timestamp', 'timestamp_modified'],
      dtype='object')
-----
1
Index(['cli

In [ ]:
result_dict = {}

# Iterate through the dictionary
for key, df_list in temp_dict.items():
    divided_dfs_list = []
    
    # Iterate through the list of dataframes
    for df in df_list:
        # Sort dataframe by 'timestamp_modified'
        df = df.sort_values(by='timestamp_modified')
        
        # Find the minimum and maximum timestamps
        min_timestamp = df['timestamp_modified'].min()
        max_timestamp = df['timestamp_modified'].max()
        
        # Divide the dataframe into 5-minute intervals (300,000 ms)
        interval_start = min_timestamp
        while interval_start <= max_timestamp:
            interval_end = interval_start + 300000
            
            # Filter rows within the 5-minute interval
            mask = (df['timestamp_modified'] >= interval_start) & (df['timestamp_modified'] < interval_end)
            divided_df = df[mask]
            
            if not divided_df.empty:
                divided_dfs_list.append(divided_df)
            
            interval_start = interval_end
            
    result_dict[key] = divided_dfs_list

In [64]:
# Check length of result_dict for each key
for key, df_lists in result_dict.items():
    for df_list in df_lists:
        print(key)
        print(len(df_list))
        print()

1
10000

1
10000

1
10000

1
10000

1
6628

1
9455

1
9828

1
9952

1
9461

1
6681

1
8439

1
4377

1
5176

1
4244

1
10000

1
8415

1
2655

1
2870

1
9174

1
6249

1
9889

1
10000

1
7654

1
756

1
1613

1
5459

1
1365

1
4307

1
6090

1
6571

1
9385

1
4508

1
8953

1
5774

1
4320

1
2661

1
306

1
2118

1
9484

1
9422

1
5871

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
9001

1
9617

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
8939

1
10000

1
10000

1
10000

1
8014

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
9187

1
10000

1
10000

1
10000

1
10000

1
452

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
9904

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
10000

1
8936

1
9902

1
10000

1
10000

1
10000

1
10000

1
10000

1
5892

1
10000

1
10000

1
10000

1
10000

1
10000

1
8793

1
9209

1
2248

1
2381

1
2330

1
5624

1
7931

1
5260

1
1

In [14]:
user_dict['1'][0]

,client_timestamp,x,y,button,state,window\n,timestamp
0,1570169899.971,694,489,None,Move,browsing\n,1570169899971
1,1570169899.99,703,473,None,Move,browsing\n,1570169899990
2,1570169900.0089998,713,454,None,Move,browsing\n,1570169900008
3,1570169900.028,725,436,None,Move,browsing\n,1570169900028
4,1570169900.046,737,417,None,Move,browsing\n,1570169900046
...,...,...,...,...,...,...,...
425866,1570232251.22,1785,822,None,Move,browsing\n,1570232251220
425867,1570232251.231,1789,820,None,Move,browsing\n,1570232251231
425868,1570232251.242,1790,819,None,Move,browsing\n,1570232251242
425869,1570232251.252,1791,818,None,Move,browsing\n,1570232251252


In [15]:
test_df = modify_timestamp(user_dict['1'][0])
test_df.head()

,client_timestamp,x,y,button,state,window\n,timestamp,timestamp_modified
0,1570169899.971,694,489,None,Move,browsing\n,1570169899971,1570169899971
1,1570169899.99,703,473,None,Move,browsing\n,1570169899990,1570169899981
2,1570169900.0089998,713,454,None,Move,browsing\n,1570169900008,1570169899991
3,1570169900.028,725,436,None,Move,browsing\n,1570169900028,1570169900001
4,1570169900.046,737,417,None,Move,browsing\n,1570169900046,1570169900011


In [17]:
# Iterate through the keys and apply the modify_timestamp function
for key in user_dict:
    user_dict[key] = [modify_timestamp(df) for df in user_dict[key]]

IndexError: single positional indexer is out-of-bounds

In [5]:
new_dict = user_dict#{key: user_dict[key] for key in ['1', '2']}

print(new_dict)

NameError: name 'user_dict' is not defined